# RoBERTa Binary Classifier for the PCL Task
This notebook trains, evaluates, and exports a RoBERTa-based binary classifier for SemEval Task 4 Subtask 1 (PCL detection) with reproducible logging, checkpointing, and threshold tuning.

**Notebook overview:** Load the SemEval PCL TSV from /data, preserve the HTML/URL/whitespace cleaning, tokenize with RoBERTa-base, and train a weighted binary classifier with logging, checkpointing/resume, and threshold tuning/export already wired up.

**Imbalance handling update**

Training now uses a `WeightedRandomSampler` to oversample positive PCL examples toward a configurable target ratio (default 45%) while the dev loader keeps the original distribution. This makes the sampling scheme the primary imbalance remedy instead of only class weighting. An optional Focal Loss (gamma=1.5) can also be toggled on to focus on hard positives; class weighting stays off by default to avoid double-counting when oversampling.

**Classifier head upgrade**

The RoBERTa encoder now feeds a two-layer GELU MLP classification head with extra dropout instead of the default single linear head. This architectural change gives the model more capacity to carve out minority-class decision boundaries while still allowing the rest of the training/evaluation/resume pipeline to save/load checkpoints as before.

In [2]:
import csv
import json
import math
import os
import random
import re
import sys
import time
import shutil
from pathlib import Path
from html import unescape
from itertools import islice

import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix
import sklearn
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, get_linear_schedule_with_warmup, RobertaModel, RobertaPreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
import transformers

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU")


/home/aaronthomas/ImperialProgramming/Fourth_Yr/NLP/CW/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.13.1 (main, Jul  5 2025, 18:17:30) [GCC 13.3.0]
PyTorch: 2.10.0+cu128
Transformers: 5.2.0
Pandas: 3.0.1
NumPy: 2.4.2
scikit-learn: 1.8.0
Using GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:

DATA_CANDIDATES = [
    Path('/data/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv'),
    Path('data/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv')
]

for candidate in DATA_CANDIDATES:
    if candidate.exists():
        DATA_PATH = candidate
        break
else:
    raise FileNotFoundError('dontpatronizeme_pcl.tsv not found under /data or ./data directories.')

print(f'Dataset path resolved to {DATA_PATH.resolve()}')

COLUMNS = ['idx', 'article_id', 'keyword', 'country', 'text', 'label']

MODEL_NAME = 'roberta-base'
MAX_LENGTH = 128
BATCH_SIZE = 16
LR = 1e-5
EPOCHS = 5  # train for 3-5 epochs as required
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP_NORM = 1.0
R_DROP_ALPHA = 0.5  # set to 0 to disable R-Drop regularization
SEED = 42
RESUME_FROM = None  #set to Path('checkpoints/epoch_k') to resume training
SAMPLING_STRATEGY = 'weighted_random'  # options: 'weighted_random', 'none'
TARGET_POSITIVE_RATIO = 0.45  # oversample positives toward this ratio in each epoch
USE_CLASS_WEIGHTS = False  # disable by default to avoid double-counting with oversampling
USE_FOCAL_LOSS = True
FOCAL_GAMMA = 1.5
MLP_HEAD_HIDDEN_MULT = 2.0  # width multiplier for the MLP classification head
MLP_HEAD_DROPOUT = 0.2  # dropout applied inside the custom head


CHECKPOINT_DIR = Path('checkpoints')
LOG_DIR = Path('logs')
BEST_MODEL_DIR = Path('bestmodel')

for path_dir in [CHECKPOINT_DIR, LOG_DIR, BEST_MODEL_DIR]:
    path_dir.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)
print(f'Random seed set to {SEED}')
print(f'R-Drop alpha: {R_DROP_ALPHA}')


Dataset path resolved to /home/aaronthomas/ImperialProgramming/Fourth_Yr/NLP/CW/BestModel/data/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv
Random seed set to 42
R-Drop alpha: 0.5


In [4]:
HTML_TAG_RE = re.compile(r'<[^>]+>')
URL_RE = re.compile(r'(https?://\S+|www\.\S+)', re.IGNORECASE)
MULTISPACE_RE = re.compile(r'\s+')


def clean_text(text: str) -> str:
    '''Decode entities, drop HTML, remove URLs, and collapse whitespace.'''
    if not isinstance(text, str):
        text = '' if text is None else str(text)
        
    text = unescape(text)
    text = re.sub(URL_RE, ' ', text)
    text = re.sub(r'<br\s*/?>', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'</?(p|div|span|strong|em)[^>]*>', ' ', text, flags=re.IGNORECASE)
    text = HTML_TAG_RE.sub(' ', text)
    text = MULTISPACE_RE.sub(' ', text)
    return text.strip()


def read_pcl_file(path: Path) -> pd.DataFrame:
    skiprows = 4
    df = pd.read_csv(
        path,
        sep='\t',
        header=None,
        names=COLUMNS,
        skiprows=skiprows,
        quoting=csv.QUOTE_NONE,
        on_bad_lines='skip',
        engine='python'
    )
    return df


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['text'] = df['text'].fillna('')
    df['clean_text'] = df['text'].apply(clean_text)
    df['binary_label'] = (df['label'] >= 2).astype(int)
    return df

In [5]:
full_df = prepare_dataframe(read_pcl_file(DATA_PATH))
print(f'Total paragraphs loaded: {len(full_df):,}')
print(full_df['binary_label'].value_counts().sort_index())


train_df, dev_df = train_test_split(
    full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=full_df['binary_label']
)

train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)


def describe_split(name: str, df: pd.DataFrame):
    counts = df['binary_label'].value_counts().sort_index()
    pct = df['binary_label'].value_counts(normalize=True).sort_index()
    print(f"{name} size: {len(df):,}")
    for label in counts.index:
        print(f"  label {label}: {counts[label]:,} ({pct[label] * 100:.2f}%)")


describe_split('Train', train_df)
describe_split('Dev', dev_df)

Total paragraphs loaded: 10,469
binary_label
0    9476
1     993
Name: count, dtype: int64
Train size: 8,375
  label 0: 7,581 (90.52%)
  label 1: 794 (9.48%)
Dev size: 2,094
  label 0: 1,895 (90.50%)
  label 1: 199 (9.50%)


In [6]:
tokenizer_source = RESUME_FROM if RESUME_FROM else MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
print(f'Tokenizer loaded from {tokenizer_source}')

all_texts = full_df['clean_text'].tolist()
token_lengths = []
for start in range(0, len(all_texts), 256):
    batch_texts = all_texts[start:start + 256]
    encodings = tokenizer(batch_texts, padding=False, truncation=False, add_special_tokens=True)
    token_lengths.extend(len(ids) for ids in encodings['input_ids'])

lengths_array = np.array(token_lengths)
p50, p95, p99 = np.percentile(lengths_array, [50, 95, 99])
print(f'Token length percentiles -> p50: {p50:.0f}, p95: {p95:.0f}, p99: {p99:.0f}')
print(f'Max observed length: {lengths_array.max():.0f}; using MAX_LENGTH={MAX_LENGTH}')

Token indices sequence length is longer than the specified maximum sequence length for this model (546 > 512). Running this sequence through the model will result in indexing errors


Tokenizer loaded from roberta-base
Token length percentiles -> p50: 48, p95: 115, p99: 160
Max observed length: 1005; using MAX_LENGTH=128


In [7]:
class PCLDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer: AutoTokenizer, max_length: int):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx: int):
        text = self.data.loc[idx, 'clean_text']
        label = int(self.data.loc[idx, 'binary_label'])
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item


def build_weighted_sampler(train_frame: pd.DataFrame, target_pos_ratio: float):
    labels = train_frame['binary_label'].values
    pos_count = int(labels.sum())
    neg_count = int(len(labels) - pos_count)
    if pos_count == 0 or neg_count == 0:
        return None, 1.0, float(pos_count / max(1, len(labels)))
    clipped_ratio = float(np.clip(target_pos_ratio, 1e-3, 0.999))
    pos_weight = (clipped_ratio * neg_count) / max(1e-6, (1 - clipped_ratio) * pos_count)
    sample_weights = torch.ones(len(labels), dtype=torch.float)
    label_mask = torch.tensor(labels == 1, dtype=torch.bool)
    sample_weights[label_mask] = float(pos_weight)
    sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(labels), replacement=True)
    expected_ratio = (pos_weight * pos_count) / (pos_weight * pos_count + neg_count)
    return sampler, float(pos_weight), float(expected_ratio)


train_dataset = PCLDataset(train_df, tokenizer, MAX_LENGTH)
dev_dataset = PCLDataset(dev_df, tokenizer, MAX_LENGTH)

train_labels_series = train_df['binary_label']
pos_count = int(train_labels_series.sum())
neg_count = int(len(train_labels_series) - pos_count)
train_pos_ratio = pos_count / max(1, len(train_labels_series))

train_sampler = None
sampler_pos_weight = 1.0
expected_sampled_ratio = train_pos_ratio
if SAMPLING_STRATEGY == 'weighted_random':
    train_sampler, sampler_pos_weight, expected_sampled_ratio = build_weighted_sampler(train_df, TARGET_POSITIVE_RATIO)
    if train_sampler is None:
        print('WeightedRandomSampler requested but not applied because only one class is present.')
    else:
        print(f'WeightedRandomSampler enabled -> target_pos_ratio={TARGET_POSITIVE_RATIO:.2f}, expected_sampled_ratio={expected_sampled_ratio:.2f}')
else:
    print('Sampling strategy: natural distribution with shuffling.')

if train_sampler is not None:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler, shuffle=False)
else:
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)

steps_per_epoch = len(train_loader)
print(f'Steps per epoch: {steps_per_epoch}')

w_pos = (neg_count / max(1, pos_count)) if pos_count > 0 else 1.0
if USE_CLASS_WEIGHTS:
    print(f'Class weights applied -> w_neg=1.0, w_pos={w_pos:.2f} (neg: {neg_count}, pos: {pos_count})')
else:
    print(f'Class weights disabled -> computed w_pos would be {w_pos:.2f} (neg: {neg_count}, pos: {pos_count})')

run_config = {
    'sampling_strategy_requested': SAMPLING_STRATEGY,
    'sampling_strategy_used': 'weighted_random' if train_sampler is not None else 'none',
    'target_positive_ratio': float(TARGET_POSITIVE_RATIO),
    'expected_sampled_positive_ratio': float(expected_sampled_ratio),
    'observed_positive_ratio': float(train_pos_ratio),
    'train_size': int(len(train_dataset)),
    'dev_size': int(len(dev_dataset)),
    'positive_train': int(pos_count),
    'negative_train': int(neg_count),
    'batch_size': int(BATCH_SIZE),
    'use_class_weights': bool(USE_CLASS_WEIGHTS),
    'class_weight_pos': float(w_pos) if USE_CLASS_WEIGHTS else None,
    'use_focal_loss': bool(USE_FOCAL_LOSS),
    'focal_gamma': float(FOCAL_GAMMA) if USE_FOCAL_LOSS else None
}
run_config_path = LOG_DIR / 'run_config.json'
with run_config_path.open('w') as f:
    json.dump(run_config, f, indent=2)
print(f'Sampling configuration logged to {run_config_path}')


WeightedRandomSampler enabled -> target_pos_ratio=0.45, expected_sampled_ratio=0.45
Steps per epoch: 524
Class weights disabled -> computed w_pos would be 9.55 (neg: 7581, pos: 794)
Sampling configuration logged to logs/run_config.json


In [ ]:
train_log_path = LOG_DIR / 'train_log.csv'
eval_log_path = LOG_DIR / 'eval_log.csv'
training_state_file = CHECKPOINT_DIR / 'training_state.json'


def append_csv_row(path: Path, fieldnames, row: dict):
    file_exists = path.exists()
    with path.open('a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def compute_metrics(probs: np.ndarray, labels: np.ndarray, threshold: float):
    preds = (probs >= threshold).astype(int)
    precision, recall, f1_pos, _ = precision_recall_fscore_support(
        labels,
        preds,
        average='binary',
        zero_division=0
    )
    macro_f1 = f1_score(labels, preds, average='macro', zero_division=0)
    cm = confusion_matrix(labels, preds)
    return {
        'threshold': float(threshold),
        'f1_pos': float(f1_pos),
        'f1_macro': float(macro_f1),
        'precision_pos': float(precision),
        'recall_pos': float(recall),
        'confusion_matrix': cm
    }


def evaluate_model(model, dataloader, criterion):
    model.eval()
    losses, probs, labels = [], [], []
    with torch.no_grad():
        for batch in dataloader:
            batch_labels = batch.pop('labels').to(device)
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            logits = outputs.logits
            loss = criterion(logits, batch_labels)
            losses.append(loss.item())
            scores = torch.softmax(logits, dim=1)[:, 1]
            probs.extend(scores.detach().cpu().numpy())
            labels.extend(batch_labels.cpu().numpy())
    probs = np.array(probs)
    labels = np.array(labels)
    metrics = compute_metrics(probs, labels, threshold=0.5)
    metrics['loss'] = float(np.mean(losses)) if losses else 0.0
    metrics['probs'] = probs
    metrics['labels'] = labels
    return metrics


def tune_threshold(probs: np.ndarray, labels: np.ndarray):
    best = None
    for threshold in np.arange(0.05, 1.0, 0.05):
        metrics = compute_metrics(probs, labels, float(threshold))
        if best is None or metrics['f1_pos'] > best['f1_pos']:
            best = metrics
    return best


def save_checkpoint(target_dir: Path, model, tokenizer, optimizer, scheduler, epoch_idx: int, global_step: int):
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=False)
    model.save_pretrained(target_dir)
    tokenizer.save_pretrained(target_dir)
    torch.save(
        {
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'epoch': epoch_idx,
            'global_step': global_step
        },
        target_dir / 'optimizer_scheduler.pt'
    )


def save_training_state(path: Path, state: dict):
    with path.open('w') as f:
        json.dump(state, f, indent=2)

def rdrop_loss(logits1, logits2, labels, base_criterion, alpha):
    base_loss = 0.5 * (base_criterion(logits1, labels) + base_criterion(logits2, labels))
    if alpha <= 0:
        return base_loss
    log_probs1 = F.log_softmax(logits1, dim=-1)
    log_probs2 = F.log_softmax(logits2, dim=-1)
    probs1 = log_probs1.exp()
    probs2 = log_probs2.exp()
    kl_ab = torch.nn.functional.kl_div(log_probs1, probs2, reduction='batchmean')
    kl_ba = torch.nn.functional.kl_div(log_probs2, probs1, reduction='batchmean')
    return base_loss + alpha * 0.5 * (kl_ab + kl_ba)

class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        probs = torch.softmax(logits, dim=-1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_factor = torch.pow(torch.clamp(1.0 - pt, min=1e-6), self.gamma)
        return (focal_factor * ce_loss).mean()


In [9]:
class RobertaMLPClassificationHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_size = config.hidden_size
        multiplier = float(getattr(config, 'mlp_head_hidden_mult', 2.0))
        intermediate = max(1, int(hidden_size * multiplier))
        dropout_prob = float(getattr(config, 'mlp_head_dropout', config.hidden_dropout_prob))
        self.dropout = nn.Dropout(dropout_prob)
        self.dense = nn.Linear(hidden_size, intermediate)
        self.activation = nn.GELU()
        self.dropout2 = nn.Dropout(dropout_prob)
        self.out_proj = nn.Linear(intermediate, config.num_labels)

    def forward(self, features, **kwargs):
        x = features[:, 0, :]
        x = self.dropout(x)
        x = self.dense(x)
        x = self.activation(x)
        x = self.dropout2(x)
        x = self.out_proj(x)
        return x


class RobertaPCLForSequenceClassification(RobertaPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.classifier = RobertaMLPClassificationHead(config)
        self.post_init()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )

        sequence_output = outputs[0]
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            if self.num_labels == 1:
                loss_fct = nn.MSELoss()
                loss = loss_fct(logits.view(-1), labels.view(-1))
            else:
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )


In [10]:
model_source = RESUME_FROM if RESUME_FROM else MODEL_NAME
config = AutoConfig.from_pretrained(model_source, num_labels=2)
if not hasattr(config, 'mlp_head_hidden_mult'):
    config.mlp_head_hidden_mult = MLP_HEAD_HIDDEN_MULT
if not hasattr(config, 'mlp_head_dropout'):
    config.mlp_head_dropout = MLP_HEAD_DROPOUT
model = RobertaPCLForSequenceClassification.from_pretrained(model_source, config=config)
model.to(device)
print(f"Custom MLP head -> multiplier={config.mlp_head_hidden_mult}, dropout={config.mlp_head_dropout}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_training_steps = max(1, len(train_loader)) * EPOCHS
warmup_steps = max(1, int(total_training_steps * WARMUP_RATIO))
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

class_weights = torch.tensor([1.0, w_pos], dtype=torch.float, device=device) if USE_CLASS_WEIGHTS else None
if USE_FOCAL_LOSS:
    criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights)

print(f"Criterion: {'FocalLoss' if USE_FOCAL_LOSS else 'CrossEntropyLoss'} | Class weights: {'on' if USE_CLASS_WEIGHTS else 'off'}")
use_rdrop = R_DROP_ALPHA > 0
print(f'R-Drop enabled: {use_rdrop} (alpha={R_DROP_ALPHA})')

use_amp = torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)

train_log_fields = ['epoch', 'step', 'train_loss', 'lr', 'rdrop_alpha']
eval_log_fields = ['epoch', 'eval_loss', 'f1_pos', 'f1_macro', 'precision_pos', 'recall_pos', 'rdrop_alpha']

start_epoch = 0
global_step = 0
best_dev_f1 = 0.0
best_epoch = -1
best_metrics = None

if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    state_path = resume_path / 'optimizer_scheduler.pt'
    if state_path.exists():
        ckpt_state = torch.load(state_path, map_location='cpu')
        optimizer.load_state_dict(ckpt_state['optimizer'])
        scheduler.load_state_dict(ckpt_state['scheduler'])
        start_epoch = ckpt_state.get('epoch', 0) + 1
        global_step = ckpt_state.get('global_step', 0)
        print(f'Resumed optimizer/scheduler from {resume_path}. Next epoch: {start_epoch + 1}, global step: {global_step}.')
    else:
        print(f'Warning: RESUME_FROM={resume_path} provided but optimizer state not found.')

    if training_state_file.exists():
        with training_state_file.open('r') as f:
            state_data = json.load(f)
        best_dev_f1 = state_data.get('best_f1', 0.0)
        best_epoch = state_data.get('best_epoch', -1)
        best_metrics = state_data.get('best_metrics')
        if best_metrics and 'confusion_matrix' in best_metrics:
            best_metrics['confusion_matrix'] = np.array(best_metrics['confusion_matrix'])
        print(f"Loaded historical best F1={best_dev_f1:.4f} from training_state.json.")
else:
    if training_state_file.exists():
        print('Note: training_state.json exists and will be overwritten during this run.')

print(f'Total training steps: {total_training_steps} | Warmup steps: {warmup_steps}')

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 411.25it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaPCLForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Cons

Custom MLP head -> multiplier=2.0, dropout=0.2
Criterion: FocalLoss | Class weights: off
R-Drop enabled: True (alpha=0.5)
Note: training_state.json exists and will be overwritten during this run.
Total training steps: 2620 | Warmup steps: 262


/tmp/ipykernel_3450/3035530328.py:32: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)


In [11]:
if len(train_loader) == 0:
    raise RuntimeError('Training set is empty; please verify the input data.')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    epoch_loss = 0.0
    progress_bar = tqdm(train_loader, total=len(train_loader), desc=f'Epoch {epoch + 1}/{EPOCHS}', leave=False)
    for step_idx, batch in enumerate(progress_bar, start=1):
        labels = batch.pop('labels').to(device)
        inputs = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        with autocast(enabled=use_amp):
            logits = model(**inputs).logits
            if use_rdrop:
                logits2 = model(**inputs).logits
                loss = rdrop_loss(logits, logits2, labels, criterion, R_DROP_ALPHA)
            else:
                loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step_loss = loss.item()
        epoch_loss += step_loss
        global_step += 1
        current_lr = scheduler.get_last_lr()[0]
        progress_bar.set_postfix(loss=f'{step_loss:.4f}', lr=f'{current_lr:.2e}')

        append_csv_row(
            train_log_path,
            train_log_fields,
            {'epoch': epoch + 1, 'step': global_step, 'train_loss': float(step_loss), 'lr': float(current_lr), 'rdrop_alpha': float(R_DROP_ALPHA)}
        )

    avg_train_loss = epoch_loss / max(1, len(train_loader))
    eval_outputs = evaluate_model(model, dev_loader, criterion)
    append_csv_row(
        eval_log_path,
        eval_log_fields,
        {
            'epoch': epoch + 1,
            'eval_loss': float(eval_outputs['loss']),
            'f1_pos': float(eval_outputs['f1_pos']),
            'f1_macro': float(eval_outputs['f1_macro']),
            'precision_pos': float(eval_outputs['precision_pos']),
            'recall_pos': float(eval_outputs['recall_pos']),
            'rdrop_alpha': float(R_DROP_ALPHA)
        }
    )

    print(f"Epoch {epoch + 1}: train_loss={avg_train_loss:.4f} | dev_loss={eval_outputs['loss']:.4f} | dev F1@0.5={eval_outputs['f1_pos']:.4f}")

    tuned_metrics = tune_threshold(eval_outputs['probs'], eval_outputs['labels'])
    print(
        f"Best threshold this epoch: {tuned_metrics['threshold']:.2f} -> F1={tuned_metrics['f1_pos']:.4f}, "
        f"Precision={tuned_metrics['precision_pos']:.4f}, Recall={tuned_metrics['recall_pos']:.4f}"
    )

    epoch_dir = CHECKPOINT_DIR / f'epoch_{epoch + 1}'
    save_checkpoint(epoch_dir, model, tokenizer, optimizer, scheduler, epoch, global_step)

    if tuned_metrics['f1_pos'] > best_dev_f1:
        best_dev_f1 = tuned_metrics['f1_pos']
        best_epoch = epoch + 1
        best_metrics = tuned_metrics
        best_dir = CHECKPOINT_DIR / 'best'
        save_checkpoint(best_dir, model, tokenizer, optimizer, scheduler, epoch, global_step)
        threshold_payload = {
            'threshold': tuned_metrics['threshold'],
            'f1_pos': tuned_metrics['f1_pos'],
            'f1_macro': tuned_metrics['f1_macro'],
            'precision_pos': tuned_metrics['precision_pos'],
            'recall_pos': tuned_metrics['recall_pos']
        }
        with (best_dir / 'threshold.json').open('w') as f:
            json.dump(threshold_payload, f, indent=2)
        print(f'New best model saved from epoch {epoch + 1} with F1={best_dev_f1:.4f}.')
    else:
        print('Dev F1 did not improve this epoch.')

    serializable_best = None
    if best_metrics is not None:
        serializable_best = {k: v for k, v in best_metrics.items() if k != 'confusion_matrix'}
        serializable_best['confusion_matrix'] = best_metrics['confusion_matrix'].tolist()

    training_state = {
        'next_epoch': epoch + 1,
        'best_f1': best_dev_f1,
        'best_epoch': best_epoch,
        'global_step': global_step,
        'best_threshold': best_metrics['threshold'] if best_metrics else None,
        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
        'best_metrics': serializable_best,
        'rdrop_alpha': R_DROP_ALPHA
    }
    save_training_state(training_state_file, training_state)

print('Training complete.')

Epoch 1/5:   0%|          | 0/524 [00:00<?, ?it/s]/tmp/ipykernel_3450/850678567.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 1: train_loss=0.1656 | dev_loss=0.0990 | dev F1@0.5=0.5820
Best threshold this epoch: 0.50 -> F1=0.5820, Precision=0.4913, Recall=0.7136


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


New best model saved from epoch 1 with F1=0.5820.


Epoch 2/5:   0%|          | 0/524 [00:00<?, ?it/s]/tmp/ipykernel_3450/850678567.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


...

Dev F1 did not improve this epoch.


Epoch 5/5:   0%|          | 0/524 [00:00<?, ?it/s]/tmp/ipykernel_3450/850678567.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 5: train_loss=0.0180 | dev_loss=0.2700 | dev F1@0.5=0.5686
Best threshold this epoch: 0.05 -> F1=0.5850, Precision=0.5331, Recall=0.6482


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.31s/it]


New best model saved from epoch 5 with F1=0.5850.
Training complete.


**R-Drop impact:** Averaging two dropout-perturbed passes and penalizing their KL divergence keeps logits consistent, which typically stabilizes precision/recall for the minority PCL class even with class-weighted loss.

In [12]:
best_checkpoint_dir = CHECKPOINT_DIR / 'best'
threshold_path = best_checkpoint_dir / 'threshold.json'
if not best_checkpoint_dir.exists():
    raise FileNotFoundError('Best checkpoint not found; ensure training has completed successfully.')

best_threshold = 0.5
if threshold_path.exists():
    with threshold_path.open('r') as f:
        threshold_info = json.load(f)
    best_threshold = threshold_info.get('threshold', best_threshold)
else:
    threshold_info = {'threshold': best_threshold}
    print('No threshold.json found in best checkpoint; defaulting to 0.5.')

best_model = RobertaPCLForSequenceClassification.from_pretrained(best_checkpoint_dir).to(device)
best_model.eval()

best_eval = evaluate_model(best_model, dev_loader, criterion)
final_metrics = compute_metrics(best_eval['probs'], best_eval['labels'], best_threshold)

print(f"Best threshold: {best_threshold:.2f}")
print(f"Dev positive-class F1: {final_metrics['f1_pos']:.4f}")
print(f"Dev macro F1: {final_metrics['f1_macro']:.4f}")
print(f"Precision (pos): {final_metrics['precision_pos']:.4f} | Recall (pos): {final_metrics['recall_pos']:.4f}")
print('Confusion matrix [[tn, fp], [fn, tp]]:')
print(final_metrics['confusion_matrix'])

Loading weights:   0%|          | 1/201 [00:00<00:00, 277.16it/s, Materializing param=classifier.dense.bias]  

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 460.72it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


Best threshold: 0.05
Dev positive-class F1: 0.5850
Dev macro F1: 0.7681
Precision (pos): 0.5331 | Recall (pos): 0.6482
Confusion matrix [[tn, fp], [fn, tp]]:
[[1782  113]
 [  70  129]]


In [13]:
if BEST_MODEL_DIR.exists():
    shutil.rmtree(BEST_MODEL_DIR)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)

best_export_model = RobertaPCLForSequenceClassification.from_pretrained(best_checkpoint_dir)
best_export_tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_dir)

best_export_model.save_pretrained(BEST_MODEL_DIR)
best_export_tokenizer.save_pretrained(BEST_MODEL_DIR)

if threshold_path.exists():
    shutil.copy(threshold_path, BEST_MODEL_DIR / 'threshold.json')

print(f"Best model saved to {BEST_MODEL_DIR.resolve()}")

Loading weights:   1%|          | 2/201 [00:00<00:00, 415.96it/s, Materializing param=classifier.dense.weight]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.02it/s]

Best model saved to /home/aaronthomas/ImperialProgramming/Fourth_Yr/NLP/CW/BestModel/bestmodel


In [20]:
demo_texts = [
    'We must uplift those poor souls who cannot help themselves.',
    'The committee released a neutral report on fiscal policy.',
    'These resilient survivors need all the help we can give them.',
    'poor people'
]

inference_tokenizer = AutoTokenizer.from_pretrained(BEST_MODEL_DIR)
inference_model = RobertaPCLForSequenceClassification.from_pretrained(BEST_MODEL_DIR).to(device)
inference_model.eval()

threshold_file = BEST_MODEL_DIR / 'threshold.json'
infer_threshold = 0.5
if threshold_file.exists():
    with threshold_file.open('r') as f:
        infer_threshold = json.load(f).get('threshold', infer_threshold)

encoded = inference_tokenizer(
    demo_texts,
    padding=True,
    truncation=True,
    max_length=MAX_LENGTH,
    return_tensors='pt'
)
encoded = {k: v.to(device) for k, v in encoded.items()}

with torch.no_grad():
    logits = inference_model(**encoded).logits
    probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

for text, prob in zip(demo_texts, probs):
    label = int(prob >= infer_threshold)
    print(f"PCL prob={prob:.3f} | pred={label} | text='{text}'")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 426.53it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              


PCL prob=0.939 | pred=1 | text='We must uplift those poor souls who cannot help themselves.'
PCL prob=0.005 | pred=0 | text='The committee released a neutral report on fiscal policy.'
PCL prob=0.947 | pred=1 | text='These resilient survivors need all the help we can give them.'
PCL prob=0.066 | pred=1 | text='poor people'
